In [1]:
!pip install scipy


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import os
import pickle

# Path to saved state directory
SAVE_DIR = "saved_analysis_state"

# LOAD PART 2 OUTPUT
with open(
    os.path.join(
        SAVE_DIR,
        "metadata",
        "part2_output.pkl"
    ),
    "rb"
) as f:
    distro_metadata = pickle.load(f)

# EXTRACT FITTING CUSTOMERS
customers_for_fitting = distro_metadata['customers_for_fitting']

work_df = distro_metadata['customer_stats_df'].copy()

work_df['eligible_for_fitting'] = (
    work_df['customer_id']
    .isin(customers_for_fitting)
)

print("Main data frame")
print(work_df.head())
print(work_df.shape)
print("="*30)
print("fitting data frame")
fit_df = work_df[
    work_df['eligible_for_fitting']
].copy()
print(fit_df.shape)
print(fit_df.columns)


Main data frame
  customer_id  n_obs              tier data_reliability  \
0     12353.0      8  basic_stats_only              low   
1     12363.0      5  basic_stats_only              low   
2     12364.0      5  basic_stats_only              low   
3     12369.0      8  basic_stats_only              low   
4     12372.0      8  basic_stats_only              low   

   recency_days_constant  recency_days_mean  recency_days_median  \
0                  False             59.750                 49.5   
1                  False             29.800                 19.0   
2                  False                NaN                  NaN   
3                  False             52.375                 38.0   
4                  False             25.750                 16.0   

  recency_days_mean_median_pattern  recency_days_std  recency_days_cv  ...  \
0                     right_skewed         56.406449         0.944041  ...   
1                     right_skewed         31.050926         1.0

In [3]:
#VARIABLE SPECIFIC DOMAIN ANALYSIS. CREATING BEHAVIOURAL SIGNAL FOR DISTRIBUTION SELECTION

# ==========================================
# RECENCY DOMAIN ANALYSIS
# ==========================================

fit_df['recency_zero_inflated'] = (
    fit_df['recency_days_zero_proportion'] > 0.05
)

fit_df['recency_exponential_like'] = (
    fit_df['recency_days_cv'].between(0.8, 1.2)
)

fit_df['recency_overdispersed'] = (
    fit_df['recency_days_cv'] > 1
)

fit_df['recency_heavy_tailed'] = (
    fit_df['recency_days_skewness'] > 2
)

fit_df['recency_light_tailed'] = (
    fit_df['recency_days_skewness'] < 2
)

fit_df['recency_extreme_tail'] = (
    fit_df['recency_days_p95_p50_ratio'] > 3
)

In [4]:
# ==========================================
# FREQUENCY DOMAIN ANALYSIS
# ==========================================

fit_df['frequency_degenerate'] = (
    fit_df['frequency_constant']
)

fit_df['frequency_overdispersed'] = (
    fit_df['frequency_cv'] > 1
)

fit_df['frequency_zero_inflated'] = (
    fit_df['frequency_zero_proportion'] > 0.05
)

fit_df['frequency_heavy_tailed'] = (
    fit_df['frequency_p95_p50_ratio'] > 3
)

# Compare observed skewness vs Poisson expectation
fit_df['poisson_expected_skew'] = (
    1 / np.sqrt(fit_df['frequency_mean'])
)

fit_df['frequency_more_skewed_than_poisson'] = (
    fit_df['frequency_skewness']
    >
    fit_df['poisson_expected_skew']
)

In [5]:
# ==========================================
# MONETARY VALUE DOMAIN ANALYSIS
# ==========================================

fit_df['monetary_heavy_tailed'] = (
    fit_df['monetary_value_skewness'] > 2
)

fit_df['monetary_overdispersed'] = (
    fit_df['monetary_value_cv'] > 1
)

fit_df['monetary_extreme_tail'] = (
    fit_df['monetary_value_p95_p50_ratio'] > 3
)

fit_df['monetary_mean_above_median'] = (
    fit_df['monetary_value_mean']
    >
    fit_df['monetary_value_median']
)

In [6]:
#Candidate distribution per customer generation
# Stores candidate distributions per customer
fit_df['recency_candidates'] = None
fit_df['frequency_candidates'] = None
fit_df['monetary_candidates'] = None

In [7]:
# RECENCY DISTRIBUTION SELECTION

def select_recency_candidates(row):

    candidates = []
    if (
        row['recency_exponential_like']
        and not row['recency_heavy_tailed']
        and not row['recency_extreme_tail']
    ):
        candidates.append('Exponential')

    if (
        row['recency_overdispersed']
        or row['recency_exponential_like']
    ):
        candidates.append('Gamma')

    
    candidates.append('Weibull')

    if (
        row['recency_heavy_tailed']
        or row['recency_extreme_tail']
    ):
        candidates.append('LogNormal')

    if (
        row['recency_heavy_tailed']
        and row['recency_extreme_tail']
        and row['n_obs'] >= 15
    ):
        candidates.append('GeneralizedPareto')

    if len(candidates) == 0:
        candidates.append('Weibull')
    return candidates


fit_df['recency_candidates'] = (
    fit_df.apply(select_recency_candidates, axis=1)
)


In [8]:
# FREQUENCY DISTRIBUTION SELECTION

def select_frequency_candidates(row):

    candidates = []

    if row['frequency_degenerate']:

        candidates.append('Degenerate')

        return candidates

    if (
        not row['frequency_overdispersed']
        and not row['frequency_more_skewed_than_poisson']
        and not row['frequency_zero_inflated']
    ):
        candidates.append('Poisson')

    if (
        row['frequency_overdispersed']
        or row['frequency_more_skewed_than_poisson']
    ):
        candidates.append('NegativeBinomial')

    if row['frequency_zero_inflated']:

        candidates.append('ZIP')
        candidates.append('ZINB')

    if (
        row['frequency_overdispersed']
        and row['frequency_heavy_tailed']
    ):
        candidates.append('Geometric')

    if (
        row['frequency_heavy_tailed']
        and row['frequency_more_skewed_than_poisson']
        and row['n_obs'] >= 15
    ):
        candidates.append('DiscreteWeibull')
    if len(candidates) == 0:
        candidates.append('NegativeBinomial')
    return candidates


fit_df['frequency_candidates'] = (
    fit_df.apply(select_frequency_candidates, axis=1)
)


In [9]:
# MONETARY DISTRIBUTION SELECTION

# ==========================================
# MONETARY DISTRIBUTION SELECTION
# CONSUMES STEP 2 BEHAVIOURAL SIGNALS
# ==========================================

def select_monetary_candidates(row):

    candidates = []

    # -----------------------------------
    # LogNormal
    # -----------------------------------
    # Default monetary baseline for
    # multiplicative spending behaviour
    # -----------------------------------
    if (
        row['monetary_mean_above_median']
        and (
            row['monetary_heavy_tailed']
            or row['monetary_extreme_tail']
        )
    ):
        candidates.append('LogNormal')

    # -----------------------------------
    # Gamma
    # -----------------------------------
    # Stable positive-support distribution
    # Better for moderate tails
    # -----------------------------------
    if (
        row['monetary_overdispersed']
        and not row['monetary_extreme_tail']
    ):
        candidates.append('Gamma')

    # -----------------------------------
    # Weibull
    # -----------------------------------
    # Flexible positive continuous family
    # Good fallback distribution
    # -----------------------------------
    candidates.append('Weibull')

    # -----------------------------------
    # Generalized Gamma
    # -----------------------------------
    # Flexible super-family containing:
    # Gamma, Weibull, LogNormal
    # -----------------------------------
    if (
        row['monetary_heavy_tailed']
        and row['monetary_overdispersed']
        and row['n_obs'] >= 15
    ):
        candidates.append('GeneralizedGamma')

    # -----------------------------------
    # Pareto
    # -----------------------------------
    # Extreme wealth/spending concentration
    # -----------------------------------
    if (
        row['monetary_extreme_tail']
        and row['monetary_heavy_tailed']
    ):
        candidates.append('Pareto')

    # -----------------------------------
    # Burr XII
    # -----------------------------------
    # Very flexible heavy-tail model
    # Reserve for highly extreme behaviour
    # -----------------------------------
    if (
        row['monetary_extreme_tail']
        and row['monetary_heavy_tailed']
        and row['monetary_overdispersed']
        and row['n_obs'] >= 20
    ):
        candidates.append('BurrXII')
    if len(candidates) == 0:
        candidates.append('Gamma')
    return candidates


fit_df['monetary_candidates'] = (
    fit_df.apply(select_monetary_candidates, axis=1)
)


In [10]:
# ==========================================
# MINIMAL CANDIDATE DISTRIBUTION SUMMARY
# ==========================================

candidate_summary = pd.DataFrame({

    # -----------------------------------
    # RECENCY
    # -----------------------------------
    'recency_exponential_pct': [
        fit_df['recency_candidates']
        .apply(lambda x: 'Exponential' in x)
        .mean()
    ],

    'recency_gamma_pct': [
        fit_df['recency_candidates']
        .apply(lambda x: 'Gamma' in x)
        .mean()
    ],

    'recency_weibull_pct': [
        fit_df['recency_candidates']
        .apply(lambda x: 'Weibull' in x)
        .mean()
    ],

    'recency_lognormal_pct': [
        fit_df['recency_candidates']
        .apply(lambda x: 'LogNormal' in x)
        .mean()
    ],

    'recency_pareto_pct': [
        fit_df['recency_candidates']
        .apply(lambda x: 'GeneralizedPareto' in x)
        .mean()
    ],

    # -----------------------------------
    # FREQUENCY
    # -----------------------------------
    'frequency_poisson_pct': [
        fit_df['frequency_candidates']
        .apply(lambda x: 'Poisson' in x)
        .mean()
    ],

    'frequency_nb_pct': [
        fit_df['frequency_candidates']
        .apply(lambda x: 'NegativeBinomial' in x)
        .mean()
    ],

    'frequency_zip_pct': [
        fit_df['frequency_candidates']
        .apply(lambda x: 'ZIP' in x)
        .mean()
    ],

    'frequency_zinb_pct': [
        fit_df['frequency_candidates']
        .apply(lambda x: 'ZINB' in x)
        .mean()
    ],

    'frequency_geometric_pct': [
        fit_df['frequency_candidates']
        .apply(lambda x: 'Geometric' in x)
        .mean()
    ],

    # -----------------------------------
    # MONETARY
    # -----------------------------------
    'monetary_lognormal_pct': [
        fit_df['monetary_candidates']
        .apply(lambda x: 'LogNormal' in x)
        .mean()
    ],

    'monetary_gamma_pct': [
        fit_df['monetary_candidates']
        .apply(lambda x: 'Gamma' in x)
        .mean()
    ],

    'monetary_weibull_pct': [
        fit_df['monetary_candidates']
        .apply(lambda x: 'Weibull' in x)
        .mean()
    ],

    'monetary_pareto_pct': [
        fit_df['monetary_candidates']
        .apply(lambda x: 'Pareto' in x)
        .mean()
    ],

    'monetary_burr_pct': [
        fit_df['monetary_candidates']
        .apply(lambda x: 'BurrXII' in x)
        .mean()
    ]

})

candidate_summary = (
    candidate_summary.T
    .reset_index()
)

candidate_summary.columns = [
    'distribution',
    'proportion'
]

candidate_summary['proportion'] = (
    candidate_summary['proportion']
    .round(3)
)

candidate_summary

,distribution,proportion
0,recency_exponential_pct,0.213
1,recency_gamma_pct,0.855
2,recency_weibull_pct,1.000
3,recency_lognormal_pct,0.442
4,recency_pareto_pct,0.001
5,frequency_poisson_pct,0.372
6,frequency_nb_pct,0.195
7,frequency_zip_pct,0.557
8,frequency_zinb_pct,0.557
9,frequency_geometric_pct,0.008


In [11]:
from scipy.optimize import root_scalar
import numpy as np
import pandas as pd

EPS = 1e-8

# ------------------------------------------
# WEIBULL MOM
# ------------------------------------------

def weibull_mom(mean_, std_):

    if pd.isna(mean_) or pd.isna(std_) or mean_ <= 0 or std_ <= 0:
        return pd.Series([np.nan, np.nan])

    cv2 = (std_ / mean_) ** 2

    def f(k):
        g1 = np.math.gamma(1 + 2/k)
        g2 = np.math.gamma(1 + 1/k)
        return (g1 / g2**2) - 1 - cv2

    try:

        upper = 20

        while f(upper) > 0 and upper < 1000:
            upper *= 2

        k = root_scalar(
            f,
            bracket=[0.1, upper],
            method='brentq'
        ).root

        lam = mean_ / np.math.gamma(1 + 1/k)

        return pd.Series([k, lam])

    except Exception:
        return pd.Series([np.nan, np.nan])

# ------------------------------------------
# RECENCY
# ------------------------------------------

fit_df[['recency_weibull_k', 'recency_weibull_lambda']] = (
    fit_df.apply(
        lambda x: weibull_mom(
            x['recency_days_mean'],
            x['recency_days_std']
        ),
        axis=1
    )
)

fit_df['recency_gamma_shape'] = np.clip(
    (
        fit_df['recency_days_mean'] ** 2
    ) / (
        fit_df['recency_days_std'] ** 2 + EPS
    ),
    0.01,
    1000
)

fit_df['recency_gamma_scale'] = (
    fit_df['recency_days_std'] ** 2
) / (
    fit_df['recency_days_mean'] + EPS
)

# ------------------------------------------
# FREQUENCY
# ------------------------------------------

fit_df['frequency_zip_lambda'] = (
    fit_df['frequency_mean']
)

fit_df['frequency_zip_pi'] = np.clip(
    fit_df['frequency_zero_proportion'],
    0,
    0.99
)

var_ = fit_df['frequency_std'] ** 2
mean_ = fit_df['frequency_mean']

valid_zinb = var_ > mean_

fit_df['frequency_zinb_r'] = np.where(
    valid_zinb,
    (mean_ ** 2) / (var_ - mean_ + EPS),
    np.nan
)

fit_df['frequency_zinb_p'] = np.where(
    valid_zinb,
    fit_df['frequency_zinb_r']
    /
    (
        fit_df['frequency_zinb_r']
        + mean_
        + EPS
    ),
    np.nan
)

# ------------------------------------------
# MONETARY
# ------------------------------------------

fit_df[['monetary_weibull_k', 'monetary_weibull_lambda']] = (
    fit_df.apply(
        lambda x: weibull_mom(
            x['monetary_value_mean'],
            x['monetary_value_std']
        ),
        axis=1
    )
)

fit_df['monetary_lognormal_sigma'] = np.nan
fit_df['monetary_lognormal_mu'] = np.nan

valid_lognormal = (
    fit_df['monetary_value_mean'] > 0
)

fit_df.loc[
    valid_lognormal,
    'monetary_lognormal_sigma'
] = np.sqrt(
    np.log(
        1 + (
            fit_df.loc[
                valid_lognormal,
                'monetary_value_std'
            ]**2
            /
            (
                fit_df.loc[
                    valid_lognormal,
                    'monetary_value_mean'
                ]**2
                + EPS
            )
        )
    )
)

fit_df.loc[
    valid_lognormal,
    'monetary_lognormal_mu'
] = (
    np.log(
        fit_df.loc[
            valid_lognormal,
            'monetary_value_mean'
        ]
    )
    -
    (
        fit_df.loc[
            valid_lognormal,
            'monetary_lognormal_sigma'
        ]**2
        / 2
    )
)
# ------------------------------------------
# FAILURE FLAGS
# ------------------------------------------

fit_df['recency_weibull_failed'] = (
    fit_df['recency_weibull_k'].isna()
)

fit_df['monetary_weibull_failed'] = (
    fit_df['monetary_weibull_k'].isna()
)

fit_df['monetary_lognormal_failed'] = (
    ~valid_lognormal
)

fit_df['frequency_zinb_failed'] = (
    ~valid_zinb
)


In [12]:
#Moment-Based Fit Evaluation
from scipy.stats import (
    weibull_min,
    gamma,
    lognorm,
    poisson,
    nbinom
)

from scipy.special import gamma as gamma_fn
EPS = 1e-8

# helper
def rel_err(obs, theo):
    return np.abs(obs - theo) / (np.abs(obs) + EPS)

def fit_score(mean_err, var_err, q_err):
    return (
        0.2 * mean_err
        + 0.3 * var_err
        + 0.5 * q_err
    )

QUANTILES = [0.25, 0.50, 0.75, 0.95]
Q_NAMES = ['p25', 'p50', 'p75', 'p95']

In [13]:
# RECONSTRUCTION FUNCTIONS


def weibull_stats(k, lam):

    if pd.isna(k) or pd.isna(lam) or k <= 0 or lam <= 0:
        return [np.nan] * 6

    mean_ = lam * gamma_fn(1 + 1/k)

    var_ = (
        lam**2
        * (
            gamma_fn(1 + 2/k)
            - gamma_fn(1 + 1/k)**2
        )
    )

    qs = weibull_min.ppf(
        QUANTILES,
        c=k,
        scale=lam
    )

    return [mean_, var_, *qs]


def gamma_stats(shape, scale):

    if (
        pd.isna(shape)
        or pd.isna(scale)
        or shape <= 0
        or scale <= 0
    ):
        return [np.nan] * 6

    mean_ = shape * scale
    var_ = shape * scale**2

    qs = gamma.ppf(
        QUANTILES,
        a=shape,
        scale=scale
    )

    return [mean_, var_, *qs]


def lognormal_stats(mu, sigma):

    if (
        pd.isna(mu)
        or pd.isna(sigma)
        or sigma <= 0
    ):
        return [np.nan] * 6

    mean_ = np.exp(mu + sigma**2 / 2)

    var_ = (
        (np.exp(sigma**2) - 1)
        * np.exp(2*mu + sigma**2)
    )

    qs = lognorm.ppf(
        QUANTILES,
        s=sigma,
        scale=np.exp(mu)
    )

    return [mean_, var_, *qs]


def zip_stats(lam, pi_):

    if (
        pd.isna(lam)
        or pd.isna(pi_)
        or lam < 0
    ):
        return [np.nan] * 6

    mean_ = (1 - pi_) * lam

    var_ = (
        (1 - pi_) * lam
        * (1 + pi_ * lam)
    )

    xs = np.arange(0, 1000)

    cdf = (
        pi_
        + (1 - pi_) * poisson.cdf(xs, lam)
    )

    qs = [
        xs[np.searchsorted(cdf, q)]
        for q in QUANTILES
    ]

    return [mean_, var_, *qs]


def zinb_stats(r, p):

    if (
        pd.isna(r)
        or pd.isna(p)
        or r <= 0
        or p <= 0
        or p >= 1
    ):
        return [np.nan] * 6

    mean_ = r * (1-p) / p

    var_ = r * (1-p) / (p**2)

    qs = nbinom.ppf(
        QUANTILES,
        r,
        p
    )

    return [mean_, var_, *qs]

In [14]:
# GENERIC EVALUATION FUNCTION


def evaluate_fit(
    df,
    prefix,
    empirical_prefix,
    stat_func,
    param_cols
):

    out_cols = [
        'theoretical_mean',
        'theoretical_var',
        'theoretical_p25',
        'theoretical_p50',
        'theoretical_p75',
        'theoretical_p95'
    ]

    df[
        [f'{prefix}_{c}' for c in out_cols]
    ] = df.apply(
        lambda x: pd.Series(
            stat_func(
                *[x[c] for c in param_cols]
            )
        ),
        axis=1
    )

    df[f'{prefix}_mean_error'] = rel_err(
        df[f'{empirical_prefix}_mean'],
        df[f'{prefix}_theoretical_mean']
    )

    df[f'{prefix}_variance_error'] = rel_err(
        df[f'{empirical_prefix}_std']**2,
        df[f'{prefix}_theoretical_var']
    )

    q_errors = []

    for q in Q_NAMES:

        empirical_col = (
            f'{empirical_prefix}_median'
            if q == 'p50'
            else f'{empirical_prefix}_{q}'
        )

        theoretical_col = (
            f'{prefix}_theoretical_{q}'
        )

        q_errors.append(
            rel_err(
                df[empirical_col],
                df[theoretical_col]
            )
        )

    df[f'{prefix}_quantile_error'] = np.mean(
        q_errors,
        axis=0
    )

    df[f'{prefix}_fit_score'] = fit_score(
        df[f'{prefix}_mean_error'],
        df[f'{prefix}_variance_error'],
        df[f'{prefix}_quantile_error']
    )

In [15]:

# RECENCY — WEIBULL

evaluate_fit(
    fit_df,
    prefix='recency_weibull',
    empirical_prefix='recency_days',
    stat_func=weibull_stats,
    param_cols=[
        'recency_weibull_k',
        'recency_weibull_lambda'
    ]
)

# RECENCY — GAMMA

evaluate_fit(
    fit_df,
    prefix='recency_gamma',
    empirical_prefix='recency_days',
    stat_func=gamma_stats,
    param_cols=[
        'recency_gamma_shape',
        'recency_gamma_scale'
    ]
)

# FREQUENCY — ZIP

evaluate_fit(
    fit_df,
    prefix='frequency_zip',
    empirical_prefix='frequency',
    stat_func=zip_stats,
    param_cols=[
        'frequency_zip_lambda',
        'frequency_zip_pi'
    ]
)

# FREQUENCY — ZINB

evaluate_fit(
    fit_df,
    prefix='frequency_zinb',
    empirical_prefix='frequency',
    stat_func=zinb_stats,
    param_cols=[
        'frequency_zinb_r',
        'frequency_zinb_p'
    ]
)

# MONETARY — WEIBULL

evaluate_fit(
    fit_df,
    prefix='monetary_weibull',
    empirical_prefix='monetary_value',
    stat_func=weibull_stats,
    param_cols=[
        'monetary_weibull_k',
        'monetary_weibull_lambda'
    ]
)
# MONETARY — LOGNORMAL
evaluate_fit(
    fit_df,
    prefix='monetary_lognormal',
    empirical_prefix='monetary_value',
    stat_func=lognormal_stats,
    param_cols=[
        'monetary_lognormal_mu',
        'monetary_lognormal_sigma'
    ]
)

In [16]:
# BEST DISTRIBUTION SELECTION

def safest_idxmin(df, cols):

    return (
        df[cols]
        .apply(
            lambda row:
            row.idxmin()
            if row.notna().any()
            else np.nan,
            axis=1
        )
        .str.replace('_fit_score', '')
    )

# -----------------------------------
# RECENCY
# -----------------------------------

fit_df['best_recency_distribution'] = (
    safest_idxmin(
        fit_df,
        [
            'recency_weibull_fit_score',
            'recency_gamma_fit_score'
        ]
    )
)

# -----------------------------------
# FREQUENCY
# -----------------------------------

fit_df['best_frequency_distribution'] = (
    safest_idxmin(
        fit_df,
        [
            'frequency_zip_fit_score',
            'frequency_zinb_fit_score'
        ]
    )
)

# -----------------------------------
# MONETARY
# -----------------------------------

fit_df['best_monetary_distribution'] = (
    safest_idxmin(
        fit_df,
        [
            'monetary_weibull_fit_score',
            'monetary_lognormal_fit_score'
        ]
    )
)

In [17]:
score_cols = [
    c for c in fit_df.columns
    if c.endswith('_fit_score')
]

fit_df[score_cols].describe()

,recency_weibull_fit_score,recency_gamma_fit_score,frequency_zip_fit_score,frequency_zinb_fit_score,monetary_weibull_fit_score,monetary_lognormal_fit_score
count,0.0,1.109000e+03,1240.000000,9.470000e+02,0.0,1237.000000
mean,NaN,5.063858e+07,0.425436,6.599790e+05,NaN,0.046674
std,NaN,6.462094e+07,0.823011,3.234985e+06,NaN,0.129287
min,NaN,5.793514e-02,0.010057,2.909091e-09,NaN,0.001601
25%,NaN,2.654070e+05,0.221881,5.910669e-02,NaN,0.014946
50%,NaN,2.976346e+07,0.321460,8.477634e-02,NaN,0.026633
75%,NaN,7.023575e+07,0.456114,1.318424e-01,NaN,0.047297
max,NaN,6.710747e+08,22.134415,2.500000e+07,NaN,3.363849


In [18]:
#Bootstrap Stability Assessment
fit_df = fit_df.copy()
B = 500
rng = np.random.default_rng(42)
def lognormal_bootstrap(mu, sigma, n):

    if (
        pd.isna(mu)
        or pd.isna(sigma)
        or sigma <= 0
        or n < 3
    ):
        return pd.Series([np.nan] * 4)

    mu_boot = []
    sigma_boot = []

    for _ in range(B):

        x = rng.lognormal(mu, sigma, int(n))

        m = np.mean(x)
        s = np.std(x)

        sigma_hat = np.sqrt(
            np.log(
                1 + s**2 / (m**2 + EPS)
            )
        )

        mu_hat = (
            np.log(m)
            - sigma_hat**2 / 2
        )

        mu_boot.append(mu_hat)
        sigma_boot.append(sigma_hat)

    mu_cv = (
        np.std(mu_boot)
        /
        (np.abs(np.mean(mu_boot)) + EPS)
    )

    sigma_cv = (
        np.std(sigma_boot)
        /
        (np.abs(np.mean(sigma_boot)) + EPS)
    )

    return pd.Series([
        mu_cv,
        sigma_cv,
        max(mu_cv, sigma_cv),
        n / 2
    ])

In [19]:
# RUN LOGNORMAL STABILITY


fit_df[
    [
        'monetary_mu_cv',
        'monetary_sigma_cv',
        'monetary_max_param_cv',
        'monetary_n_obs_per_param'
    ]
] = fit_df.apply(

    lambda x: lognormal_bootstrap(
        x['monetary_lognormal_mu'],
        x['monetary_lognormal_sigma'],
        x['n_obs']
    ),

    axis=1
)

In [20]:
# ZIP BOOTSTRAP

def zip_bootstrap(lam, pi_, n):

    if (
        pd.isna(lam)
        or pd.isna(pi_)
        or n < 3
    ):
        return pd.Series([np.nan] * 4)

    lam_boot = []
    pi_boot = []

    for _ in range(B):

        structural_zero = rng.binomial(
            1,
            pi_,
            int(n)
        )

        counts = rng.poisson(
            lam,
            int(n)
        )

        x = np.where(
            structural_zero == 1,
            0,
            counts
        )

        lam_boot.append(np.mean(x))
        pi_boot.append(np.mean(x == 0))

    lam_cv = (
        np.std(lam_boot)
        /
        (np.abs(np.mean(lam_boot)) + EPS)
    )

    pi_cv = (
        np.std(pi_boot)
        /
        (np.abs(np.mean(pi_boot)) + EPS)
    )

    return pd.Series([
        lam_cv,
        pi_cv,
        max(lam_cv, pi_cv),
        n / 2
    ])

In [21]:
# RUN ZIP STABILITY

fit_df[
    [
        'frequency_lambda_cv',
        'frequency_pi_cv',
        'frequency_max_param_cv',
        'frequency_n_obs_per_param'
    ]
] = fit_df.apply(

    lambda x: zip_bootstrap(
        x['frequency_zip_lambda'],
        x['frequency_zip_pi'],
        x['n_obs']
    ),

    axis=1
)

In [23]:
fit_df[
    [
        'monetary_max_param_cv',
        'frequency_max_param_cv'
    ]
].describe()

,monetary_max_param_cv,frequency_max_param_cv
count,1969.000000,1972.000000
mean,0.171102,1.525730
std,0.093849,3.324689
min,0.134345,0.014294
25%,0.155242,0.342831
50%,0.166640,0.568909
75%,0.180323,0.965335
max,4.178473,22.336633


In [24]:
fit_df.columns.tolist()


['customer_id',
 'n_obs',
 'tier',
 'data_reliability',
 'recency_days_constant',
 'recency_days_mean',
 'recency_days_median',
 'recency_days_mean_median_pattern',
 'recency_days_std',
 'recency_days_cv',
 'recency_days_skewness',
 'recency_days_zero_proportion',
 'recency_days_positive_proportion',
 'recency_days_min',
 'recency_days_max',
 'recency_days_range',
 'recency_days_p25',
 'recency_days_p75',
 'recency_days_p95',
 'recency_days_mean_ci_low',
 'recency_days_mean_ci_high',
 'recency_days_mean_uncertainty_pct',
 'recency_days_interpretation',
 'frequency_constant',
 'frequency_mean',
 'frequency_median',
 'frequency_mean_median_pattern',
 'frequency_std',
 'frequency_cv',
 'frequency_skewness',
 'frequency_zero_proportion',
 'frequency_positive_proportion',
 'frequency_min',
 'frequency_max',
 'frequency_range',
 'frequency_p25',
 'frequency_p75',
 'frequency_p95',
 'frequency_mean_ci_low',
 'frequency_mean_ci_high',
 'frequency_mean_uncertainty_pct',
 'frequency_interpretati

In [25]:
from scipy.stats import poisson
from scipy.stats import entropy

from sklearn.preprocessing import StandardScaler
# ONLY STABLE DISTRIBUTIONS:
#   - frequency  -> ZIP
#   - monetary   -> Lognormal
# recency handled separately 

feature_df = fit_df.copy()

# FREQUENCY (ZIP)
# fitted mean
feature_df['freq_mean_fit'] = (
    (1 - feature_df['frequency_zip_pi']) *
    feature_df['frequency_zip_lambda']
)

# fitted CV
freq_var = (
    (1 - feature_df['frequency_zip_pi']) *
    feature_df['frequency_zip_lambda'] *
    (
        1 +
        feature_df['frequency_zip_pi'] *
        feature_df['frequency_zip_lambda']
    )
)

feature_df['freq_cv_fit'] = (
    np.sqrt(freq_var) /
    feature_df['freq_mean_fit'].replace(0, np.nan)
)

# fitted skewness
feature_df['freq_skew_fit'] = (
    1 / np.sqrt(
        feature_df['frequency_zip_lambda']
        .replace(0, np.nan)
    )
)

# entropy
def zip_entropy(lam, pi, max_k=30):

    k = np.arange(max_k + 1)

    probs = np.zeros(len(k))

    probs[0] = (
        pi +
        (1 - pi) * poisson.pmf(0, lam)
    )

    probs[1:] = (
        (1 - pi) *
        poisson.pmf(k[1:], lam)
    )

    probs = probs / probs.sum()

    return entropy(probs)

feature_df['freq_entropy'] = feature_df.apply(
    lambda x: zip_entropy(
        x['frequency_zip_lambda'],
        x['frequency_zip_pi']
    ),
    axis=1
)

# shape params
feature_df['freq_lambda'] = feature_df['frequency_zip_lambda']
feature_df['freq_pi'] = feature_df['frequency_zip_pi']

# fit quality
feature_df['freq_fit_score'] = (
    feature_df['frequency_zip_fit_score']
)

# zero probability
feature_df['freq_zero_prob'] = (
    feature_df['frequency_zip_pi']
    +
    (
        1 - feature_df['frequency_zip_pi']
    ) *
    np.exp(-feature_df['frequency_zip_lambda'])
)

# bootstrap stability
feature_df['freq_param_cv'] = (
    feature_df['frequency_max_param_cv']
)

# MONETARY (LOGNORMAL)

mu = feature_df['monetary_lognormal_mu']
sigma = feature_df['monetary_lognormal_sigma']

# fitted mean
feature_df['mon_mean_fit'] = (
    np.exp(mu + (sigma**2 / 2))
)

# fitted CV
feature_df['mon_cv_fit'] = (
    np.sqrt(np.exp(sigma**2) - 1)
)

# fitted skewness
feature_df['mon_skew_fit'] = (
    (
        np.exp(sigma**2) + 2
    ) *
    np.sqrt(np.exp(sigma**2) - 1)
)

# entropy
feature_df['mon_entropy'] = (
    mu +
    0.5 +
    np.log(
        sigma * np.sqrt(2 * np.pi)
    )
)

# shape params
feature_df['mon_mu'] = mu
feature_df['mon_sigma'] = sigma

# fit quality
feature_df['mon_fit_score'] = (
    feature_df['monetary_lognormal_fit_score']
)

# bootstrap stability
feature_df['mon_param_cv'] = (
    feature_df['monetary_max_param_cv']
)

# RECENCY
# no stable distribution
# use empirical summary stats only

recency_features = [
    'recency_days_mean',
    'recency_days_cv',
    'recency_days_skewness',
    'recency_days_p95_p50_ratio',
    'recency_days_mean_uncertainty_pct'
]


feature_cols = [

    # recency empirical
    *recency_features,

    # frequency ZIP
    'freq_mean_fit',
    'freq_cv_fit',
    'freq_skew_fit',
    'freq_entropy',
    'freq_lambda',
    'freq_pi',
    'freq_fit_score',
    'freq_zero_prob',
    'freq_param_cv',

    # monetary lognormal
    'mon_mean_fit',
    'mon_cv_fit',
    'mon_skew_fit',
    'mon_entropy',
    'mon_mu',
    'mon_sigma',
    'mon_fit_score',
    'mon_param_cv'
]

X = feature_df[feature_cols].copy()


X = X.replace([np.inf, -np.inf], np.nan)

X = X.fillna(X.median())


scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print(X_scaled.shape)

X_scaled.head()


(1977, 22)


,recency_days_mean,recency_days_cv,recency_days_skewness,recency_days_p95_p50_ratio,recency_days_mean_uncertainty_pct,freq_mean_fit,freq_cv_fit,freq_skew_fit,freq_entropy,freq_lambda,...,freq_zero_prob,freq_param_cv,mon_mean_fit,mon_cv_fit,mon_skew_fit,mon_entropy,mon_mu,mon_sigma,mon_fit_score,mon_param_cv
754,-0.308841,-0.240285,-0.204311,-0.166453,-0.259212,-0.410279,-0.125616,-0.033992,-0.241592,-0.395990,...,0.240052,-0.333741,0.921008,-0.086048,-0.022519,-0.075405,1.200603,-0.810544,-0.12212,0.299082
763,2.345755,-0.894428,-1.334086,-0.166453,-0.914644,-0.587752,5.085237,4.995074,-2.957120,-0.612981,...,3.145369,0.766562,3.800850,-0.020949,-0.022501,2.140811,2.373266,0.069640,-0.12212,0.243039
764,0.069737,-0.363109,-0.350365,-0.166453,-0.238977,-0.505841,-0.018392,0.362977,-1.009026,-0.508694,...,1.038505,-0.372321,0.136646,-0.077289,-0.022517,-0.277407,0.526321,-0.690773,-0.12212,0.074375
765,-0.522985,-0.253056,-0.416915,-0.166453,-0.151445,-0.413105,-0.120883,-0.032691,-0.254535,-0.396606,...,0.280228,-0.342471,0.479224,-0.053612,-0.022511,0.521706,0.854105,-0.368654,-0.12212,-0.103769
768,-0.478628,-0.370272,-0.549955,-0.166453,-0.384520,-0.444031,-0.097662,0.055450,-0.450384,-0.433070,...,0.449807,-0.346288,-0.194350,-0.055488,-0.022511,-0.097693,0.088179,-0.394067,-0.12212,0.010897


In [26]:
X_scaled.to_parquet(
    "saved_analysis_state/data/scaled_features.parquet",
    index=False
)